# Explore the nucleus embeddings (UNI2-h)

Diagnostics behind the post-hoc classifier on the curated features from
`scripts/curate_classifier_data.py` (`data/classifier/uni2h_fold2.npz`):
embedding spaces (by class **and** tissue), separability, per-tissue metrics,
side-by-side confusion for different label types, and a retrieval demo over real
crops with their segmentation boundaries.


## Load

In [ ]:
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
from collections import Counter
from vlm_medseg.classify.dataset import CropDataset
from vlm_medseg.constants import CLASS_COLORS, TISSUE_TYPES

def _find(rel):
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / rel).exists():
            return base / rel
    raise FileNotFoundError(rel + " (run curate_classifier_data.py first)")

data = CropDataset.load(_find("data/classifier/uni2h_fold2.npz"))
X, y, tissue, split = data.features, data.class_id, data.tissue, data.split
CLS = data.class_names
print(f"{X.shape[0]} nuclei · dim={X.shape[1]} · encoder={data.encoder}")
print("split:", dict(Counter(split.tolist())))
print("class:", {CLS[c]: int((y == c).sum()) for c in range(len(CLS))})

## Class & tissue balance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cc = [int((y == c).sum()) for c in range(len(CLS))]
ax[0].bar(CLS, cc, color=[np.array(CLASS_COLORS[c]) / 255 for c in range(len(CLS))])
ax[0].set_title("nuclei per class"); ax[0].tick_params(axis="x", rotation=30)
tc = Counter(tissue.tolist())
ax[1].bar([TISSUE_TYPES[t] for t in sorted(tc)], [tc[t] for t in sorted(tc)], color="#888")
ax[1].set_title("nuclei per tissue"); ax[1].tick_params(axis="x", rotation=90, labelsize=8)
plt.tight_layout(); plt.show()

## Embedding space — side by side: by class | by tissue
Standardize → PCA(50) → UMAP (t-SNE fallback). Same projection, colored by nucleus class (left) and tissue (right), with silhouette scores for both.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(X)
Xp = PCA(n_components=min(50, X.shape[1]), random_state=0).fit_transform(Xs)
try:
    import umap
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(Xp); proj = "UMAP"
except Exception:
    from sklearn.manifold import TSNE
    emb = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(Xp); proj = "t-SNE"

fig, ax = plt.subplots(1, 2, figsize=(15, 6.2), constrained_layout=True)
for c in range(len(CLS)):
    m = y == c
    ax[0].scatter(emb[m, 0], emb[m, 1], s=7, alpha=0.6,
                  color=np.array(CLASS_COLORS[c]) / 255, label=CLS[c])
ax[0].legend(markerscale=2, fontsize=8); ax[0].set_title(f"{proj} — by nucleus class")
tcols = plt.cm.tab20(np.linspace(0, 1, len(TISSUE_TYPES)))
for t in sorted(np.unique(tissue)):
    m = tissue == t
    ax[1].scatter(emb[m, 0], emb[m, 1], s=7, alpha=0.6, color=tcols[t], label=TISSUE_TYPES[t])
ax[1].legend(markerscale=2, fontsize=6, ncol=2, loc="center left", bbox_to_anchor=(1.01, 0.5))
ax[1].set_title(f"{proj} — by tissue")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.show()
print("silhouette  by class :", round(float(silhouette_score(Xs, y, sample_size=min(3000, len(y)), random_state=0)), 3))
print("silhouette  by tissue:", round(float(silhouette_score(Xs, tissue, sample_size=min(3000, len(tissue)), random_state=0)), 3))

## Class separability
Cosine similarity between class-mean embeddings — bright off-diagonal ⇒ confusable classes.

In [ ]:
from numpy.linalg import norm
cent = np.stack([X[y == c].mean(0) for c in range(len(CLS))])
sim = cent @ cent.T / np.outer(norm(cent, axis=1), norm(cent, axis=1))
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(sim, cmap="viridis")
ax.set_xticks(range(len(CLS))); ax.set_yticks(range(len(CLS)))
ax.set_xticklabels(CLS, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(CLS, fontsize=8)
for i in range(len(CLS)):
    for j in range(len(CLS)):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if sim[i, j] < sim.max() * 0.7 else "black")
ax.set_title("class-centroid cosine similarity"); fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## kNN purity
Fraction of each nucleus's 5 nearest neighbours sharing its class — model-free local structure.

In [ ]:
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=6).fit(Xs)
_, idx = nn.kneighbors(Xs)
purity = float(np.mean([(y[idx[i, 1:]] == y[i]).mean() for i in range(len(y))]))
per_class = {CLS[c]: round(float(np.mean([(y[idx[i,1:]]==y[i]).mean() for i in np.where(y==c)[0]])), 3)
             for c in range(len(CLS))}
print("kNN(5) label purity:", round(purity, 3)); print("per class:", per_class)

## Probe sanity — linear vs MLP
Both heads on these features; the **MLP** (repo default) is then used for the confusion / per-tissue cells below.

In [ ]:
from vlm_medseg.classify.heads import build_head, DEFAULT_HEAD
from sklearn.metrics import accuracy_score, f1_score
tr, te = split == "train", split == "test"
for h in ("linear", "mlp"):
    c = build_head(h, seed=0).fit(X[tr], y[tr])
    pr = c.predict(X[te])
    print(f"[{h:6s}] test acc={accuracy_score(y[te], pr):.3f}  "
          f"macroF1={f1_score(y[te], pr, average='macro', labels=range(len(CLS)), zero_division=0):.3f}")
clf = build_head(DEFAULT_HEAD, seed=0).fit(X[tr], y[tr])  # default head -> used downstream
print("downstream head:", DEFAULT_HEAD)

## Confusion matrices — class vs tissue (side by side)
Held-out confusion for two probes on the *same* UNI2-h features: nucleus class (5) and tissue (19). Different label types — shows what the encoder separates well.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

tr = split == "train"; held = np.isin(split, ["val", "test"])
pc = clf.predict(X[held])
cm_cls = confusion_matrix(y[held], pc, labels=range(len(CLS))); acc_cls = accuracy_score(y[held], pc)
clf_t = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced")).fit(X[tr], tissue[tr])
pt = clf_t.predict(X[held])
cm_tis = confusion_matrix(tissue[held], pt, labels=range(len(TISSUE_TYPES))); acc_tis = accuracy_score(tissue[held], pt)

fig, ax = plt.subplots(1, 2, figsize=(16, 6.8))
im0 = ax[0].imshow(cm_cls, cmap="Blues")
ax[0].set_xticks(range(len(CLS))); ax[0].set_yticks(range(len(CLS)))
ax[0].set_xticklabels(CLS, rotation=45, ha="right", fontsize=8); ax[0].set_yticklabels(CLS, fontsize=8)
ax[0].set_title(f"nucleus class · acc={acc_cls:.2f}"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("ground truth")
for i in range(len(CLS)):
    for j in range(len(CLS)):
        ax[0].text(j, i, int(cm_cls[i, j]), ha="center", va="center", fontsize=8,
                   color="white" if cm_cls[i, j] > cm_cls.max() / 2 else "black")
im1 = ax[1].imshow(cm_tis, cmap="Blues")
ax[1].set_xticks(range(len(TISSUE_TYPES))); ax[1].set_yticks(range(len(TISSUE_TYPES)))
ax[1].set_xticklabels(TISSUE_TYPES, rotation=90, fontsize=6); ax[1].set_yticklabels(TISSUE_TYPES, fontsize=6)
ax[1].set_title(f"tissue · acc={acc_tis:.2f}"); ax[1].set_xlabel("predicted"); ax[1].set_ylabel("ground truth")
fig.colorbar(im0, ax=ax[0], fraction=0.046); fig.colorbar(im1, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

## Per-tissue metrics
Class composition per organ, and the probe's held-out accuracy per organ.

In [ ]:
present = sorted(np.unique(tissue))
names = [TISSUE_TYPES[t] for t in present]
comp = np.array([[(y[tissue == t] == c).mean() for c in range(len(CLS))] for t in present])
fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(present))
for c in range(len(CLS)):
    ax.bar(names, comp[:, c], bottom=bottom, color=np.array(CLASS_COLORS[c]) / 255, label=CLS[c])
    bottom += comp[:, c]
ax.set_ylabel("class fraction"); ax.set_ylim(0, 1); ax.set_title("nucleus-class composition per tissue")
ax.legend(fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.14))
ax.tick_params(axis="x", rotation=90, labelsize=8)
plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd
held = np.isin(split, ["val", "test"])
pred = clf.predict(X[held]); yt, tt = y[held], tissue[held]
rows = [(TISSUE_TYPES[t], int((tt == t).sum()), float((pred[tt == t] == yt[tt == t]).mean()))
        for t in sorted(np.unique(tt))]
dft = pd.DataFrame(rows, columns=["tissue", "n_heldout", "accuracy"]).sort_values("accuracy")
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(dft["tissue"], dft["accuracy"], color="#4c72b0")
ax.set_ylim(0, 1.05); ax.set_ylabel("probe accuracy (held-out)")
ax.set_title("per-tissue nucleus-classification accuracy"); ax.tick_params(axis="x", rotation=90, labelsize=8)
for i, acc in enumerate(dft["accuracy"]):
    ax.text(i, acc + 0.02, f"{acc:.2f}", ha="center", fontsize=7)
plt.tight_layout(); plt.show()
dft.round(3)

## Real crops & UNI2-h retrieval
Re-decode a few fold2 patches into nucleus context crops — **what UNI2-h sees** — with the **segmentation boundary in yellow**. Then a retrieval demo: query crops + their nearest neighbours in UNI2-h space, titled with the retrieved class and cosine (green = same class as query, red = different).

In [ ]:
from vlm_medseg.data import get_dataset
from vlm_medseg.classify.crops import instance_crop_and_mask
from vlm_medseg.classify.encoders import build_encoder

ds = get_dataset("pannuke", fold="fold2")
pool_crops, pool_masks, pool_y, show = [], [], [], {c: [] for c in range(len(CLS))}
for i in ds.stratified_indices(40, seed=0):
    s = ds.decode(i)
    for inst_id, c in s.inst_classes.items():
        crop, mcrop = instance_crop_and_mask(s.image, s.inst_map == inst_id, margin=data.margin, min_size=data.min_size)
        pool_crops.append(crop); pool_masks.append(mcrop); pool_y.append(int(c))
        if len(show[c]) < 6:
            show[c].append((crop, mcrop))
pool_y = np.array(pool_y)
print(f"pool: {len(pool_crops)} crops; embedding with {data.encoder} ...")
enc = build_encoder(data.encoder, device="auto")
pool_feat = enc.embed(pool_crops, progress=True)

In [ ]:
from vlm_medseg.viz import draw_boundary
fig, axes = plt.subplots(len(CLS), 6, figsize=(10, 1.7 * len(CLS)))
for r, c in enumerate(range(len(CLS))):
    for col in range(6):
        axes[r, col].axis("off")
        if col < len(show[c]):
            crop, mcrop = show[c][col]
            axes[r, col].imshow(draw_boundary(crop, mcrop))
            axes[r, col].set_title(CLS[c], fontsize=7, color=np.array(CLASS_COLORS[c]) / 255)
fig.suptitle("nucleus context crops + segmentation boundary (yellow) — what UNI2-h sees", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
from vlm_medseg.viz import draw_boundary
F = pool_feat / np.linalg.norm(pool_feat, axis=1, keepdims=True)
rng = np.random.default_rng(0)
queries = rng.choice(len(pool_crops), 5, replace=False)
K = 5
fig, axes = plt.subplots(len(queries), K + 1, figsize=(1.5 * (K + 1), 1.7 * len(queries)))
for r, q in enumerate(queries):
    sims = F @ F[q]
    nbrs = [j for j in np.argsort(-sims) if j != q][:K]
    axes[r, 0].imshow(draw_boundary(pool_crops[q], pool_masks[q])); axes[r, 0].axis("off")
    axes[r, 0].set_title(f"query\n{CLS[pool_y[q]]}", fontsize=7)
    for col, j in enumerate(nbrs, start=1):
        axes[r, col].imshow(draw_boundary(pool_crops[j], pool_masks[j])); axes[r, col].axis("off")
        same = pool_y[j] == pool_y[q]
        axes[r, col].set_title(f"{CLS[pool_y[j]]}\n{sims[j]:.2f}", fontsize=7, color="green" if same else "red")
fig.suptitle("UNI2-h retrieval — query + 5 nearest neighbours (class · cosine); yellow = segmentation boundary", fontsize=11)
plt.tight_layout(); plt.show()
p_at_5 = float(np.mean([(pool_y[[j for j in np.argsort(-(F @ F[q])) if j != q][:5]] == pool_y[q]).mean()
                        for q in range(len(pool_crops))]))
print("retrieval precision@5 (same-class neighbours):", round(p_at_5, 3))

## Takeaways
- Embedding space, side by side: clustering **by class** ⇒ the probe works; clustering **by tissue** ⇒ the encoder also carries organ context.
- Side-by-side confusion (class vs tissue) shows which label type UNI2-h separates better, and *which* classes/organs blur.
- Per-tissue accuracy flags hard organs; retrieval precision@5 + green/red neighbours show same-class retrieval concretely.
- Rare classes (Dead) stay bounded by sample count — raise `--n` / drop `--max-per-class` in curation.
